# Gene Graph from Reactome Pathway Network

## 1. Objective
We construct a gene-level graph from pathway relationships.

$$
(1) \quad G = (V, E)
$$

Where:
- $V$ = genes
- $E$ = edges induced by pathway structure



**Explanation.**  
Edges are created only between genes belonging to **related pathways**.

---

## **3. Mathematical Formulation**

### **(1) Gene Pair p-value (Per Cancer)**

$$
(1)\quad p_{ij}^{(c)} = \sqrt{ fdr_{i,c} \cdot fdr_{j,c} }
$$

**Explanation.**  
- Uses geometric mean  
- Penalizes if either gene is weak  
- Symmetric and numerically stable  

---

### **(2) Significance (Per Cancer)**

$$
(2)\quad
\text{significance}_{ij}^{(c)} =
\begin{cases}
\text{significant} & \text{if } sig_{i,c} = 1 \land sig_{j,c} = 1 \\
\text{non-significant} & \text{otherwise}
\end{cases}
$$

**Explanation.**  
Strict AND rule:
- both genes must be significant
- reduces false positives

---

### **(3) Numerical Stability**

$$
(3)\quad p_{ij}^{(c)} \leftarrow \max\left(p_{ij}^{(c)}, 10^{-300}\right)
$$

**Explanation.**  
Prevents underflow and downstream numerical issues.

---

## **4. Implementation**



In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Load expression data
# -----------------------------
expr = pd.read_csv("fdr/expression_final.csv")

# Use gene as index for FAST lookup
expr = expr.set_index("gene")

# -----------------------------
# Load and FIX mapping (wide → long)
# -----------------------------
mapping = pd.read_csv(
    "../data/processed/pathways_mapped.tsv",
    sep="\t"
)

# Rename first column
mapping = mapping.rename(columns={"PathwayID": "Pathway"})

# Convert wide → long
mapping_long = mapping.melt(
    id_vars="Pathway",
    var_name="Gene_col",
    value_name="Gene"
)

# Clean
mapping_long = mapping_long.dropna(subset=["Gene"])
mapping_long = mapping_long[mapping_long["Gene"] != ""]

# -----------------------------
# Build pathway → gene mapping
# -----------------------------
pathway_to_genes = (
    mapping_long.groupby("Pathway")["Gene"]
    .apply(set)
    .to_dict()
)

valid_genes = set(mapping_long["Gene"].unique())

print("✅ Total pathways:", len(pathway_to_genes))
print("✅ Total mapped genes:", len(valid_genes))

# -----------------------------
# Load pathway relations
# -----------------------------
relations = pd.read_csv(
    "../data/processed/ReactomePathwaysRelation_filtered.tsv",
    sep="\t",
    header=None
)
relations.columns = ["PathwayA", "PathwayB"]

# -----------------------------
# Extract cancers
# -----------------------------
cancers = [c.replace("fdr_", "") for c in expr.columns if c.startswith("fdr_")]

print("Cancers:", cancers)

# -----------------------------
# Build graph PER cancer
# -----------------------------
for cancer in cancers:

    edges = []

    fdr_col = f"fdr_{cancer}"
    sig_col = f"sig_{cancer}"

    # Skip if columns missing (safety)
    if fdr_col not in expr.columns or sig_col not in expr.columns:
        print(f"⚠️ Skipping {cancer} (missing columns)")
        continue

    for _, rel in relations.iterrows():
        pwA, pwB = rel["PathwayA"], rel["PathwayB"]

        genesA = pathway_to_genes.get(pwA, set())
        genesB = pathway_to_genes.get(pwB, set())

        if not genesA or not genesB:
            continue

        for g1 in genesA:
            if g1 not in expr.index:
                continue

            row1 = expr.loc[g1]

            for g2 in genesB:
                if g2 not in expr.index:
                    continue

                row2 = expr.loc[g2]

                fdr1 = row1[fdr_col]
                fdr2 = row2[fdr_col]

                sig1 = row1[sig_col]
                sig2 = row2[sig_col]

                # Skip invalid
                if pd.isna(fdr1) or pd.isna(fdr2):
                    continue

                # -----------------------------
                # p-value (stable)
                # -----------------------------
                pval = np.sqrt(fdr1 * fdr2)
                pval = max(pval, 1e-300)

                # -----------------------------
                # significance
                # -----------------------------
                significance = (
                    "significant" if (sig1 == 1 and sig2 == 1)
                    else "non-significant"
                )

                edges.append([
                    pwA,
                    g1,
                    pwB,
                    g2,
                    pval,
                    significance
                ])


    # # -----------------------------
    # # Save output
    # # -----------------------------
    # edges_df = pd.DataFrame(edges, columns=[
    #     "PathwayA",
    #     "Gene1",
    #     "PathwayB",
    #     "Gene2",
    #     "pvalue",
    #     "significance"
    # ])

    # out_file = f"graph_{cancer}.csv"
    # edges_df.to_csv(out_file, index=False)

    from pathlib import Path

    # -----------------------------
    # Save output
    # -----------------------------

    edges_df = pd.DataFrame(
        edges,
        columns=[
            "PathwayA",
            "Gene1",
            "PathwayB",
            "Gene2",
            "pvalue",
            "significance"
        ]
    )

    # Create output directory if it does not exist
    OUTPUT_DIR = Path("../data/reactome")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Save cancer-specific graph
    out_file = OUTPUT_DIR / f"graph_{cancer}.csv"

    edges_df.to_csv(out_file, index=False)

    print(f"Saved: {out_file}")
    
    print(f"✅ {cancer}: {edges_df.shape}")

/var/folders/z_/4_txl3tn61g8nprq0_s7tsbc0000gn/T/ipykernel_17663/2259981963.py:15: DtypeWarning: Columns (340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,56

✅ Total pathways: 2719
✅ Total mapped genes: 9217
Cancers: ['BLCA', 'BRCA', 'CHOL', 'COAD', 'ESCA', 'GBM', 'KICH', 'LIHC', 'LUAD', 'LUSC', 'PAAD', 'PRAD', 'READ', 'SARC', 'SKCM', 'STAD', 'THCA']


In [ ]:
import igraph as ig
import leidenalg

# -----------------------------
# Run Leiden per cancer
# -----------------------------
for cancer in cancers:

    file = f"graph_{cancer}.csv"

    edges_df = pd.read_csv(file)

    if edges_df.empty:
        print(f"⚠️ {cancer}: empty graph")
        continue

    # -----------------------------
    # Build gene-gene edges
    # -----------------------------
    edges_df["weight"] = -np.log10(edges_df["pvalue"].clip(lower=1e-300))
    edges_df["weight"] = edges_df["weight"].clip(lower=0)

    gene_edges = edges_df[["Gene1", "Gene2", "weight"]]

    # -----------------------------
    # Build igraph
    # -----------------------------
    genes = pd.unique(gene_edges[["Gene1", "Gene2"]].values.ravel())

    gene_to_idx = {g: i for i, g in enumerate(genes)}

    edge_list = [
        (gene_to_idx[row["Gene1"]], gene_to_idx[row["Gene2"]])
        for _, row in gene_edges.iterrows()
    ]

    weights = gene_edges["weight"].values

    g = ig.Graph(edges=edge_list, directed=False)
    g.vs["name"] = genes

    # -----------------------------
    # Run Leiden
    # -----------------------------
    partition = leidenalg.find_partition(
        g,
        leidenalg.RBConfigurationVertexPartition,
        weights=weights,
        resolution_parameter=1.0
    )

    # -----------------------------
    # Save clusters
    # -----------------------------
    clusters = []

    for cid, cluster in enumerate(partition):
        for node in cluster:
            clusters.append([genes[node], cid])

    cluster_df = pd.DataFrame(clusters, columns=["Gene", "Cluster"])

    out_file = f"leiden_clusters_{cancer}.csv"
    cluster_df.to_csv(out_file, index=False)

    print(f"✅ {cancer}: {len(partition)} clusters")

## 2. Mathematical Formulation

### Pathway mapping
$$
(2) \quad P_i \rightarrow \{g_1, g_2, ..., g_k\}
$$

### Pathway relations
$$
(3) \quad (P_i, P_j) \in R
$$

### Gene edge construction
$$
(4) \quad E = \{(g_a, g_b) \mid g_a \in P_i, g_b \in P_j\}
$$

### Weighted edges
$$
(5) \quad w(g_a, g_b) = -\log_{10}(p_{P_i})
$$


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from itertools import product


## 3. Load Data

In [ ]:
enrichment = pd.read_csv("../pathway_enrichment/results/enrichment/enrichment_simple.csv")
mapping = pd.read_csv("../data/processed/pathways_mapped.tsv", sep='\t')
relations = pd.read_csv("../data/processed/ReactomePathwaysRelation_filtered.tsv", sep='\t', header=None)
relations.columns = ['PathwayA', 'PathwayB']

/var/folders/z_/4_txl3tn61g8nprq0_s7tsbc0000gn/T/ipykernel_1521/1669800742.py:2: DtypeWarning: Columns (340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,

## 4. Build Pathway → Gene Mapping

In [ ]:
pathway_to_genes = {}
for _, row in mapping.iterrows():
    genes = [g for g in row[1:] if pd.notna(g)]
    if genes:
        pathway_to_genes[row['PathwayID']] = genes


## 5. Build Gene Graph

In [ ]:
# enrichment_dict = enrichment.set_index('stId')['p_value'].to_dict()

# G = nx.Graph()

# for _, row in relations.iterrows():
#     pA, pB = row['PathwayA'], row['PathwayB']

#     if pA not in pathway_to_genes or pB not in pathway_to_genes:
#         continue

#     genesA = pathway_to_genes[pA]
#     genesB = pathway_to_genes[pB]

#     weight = -np.log10(enrichment_dict.get(pA, 1.0) + 1e-10)

#     for gA, gB in product(genesA, genesB):
#         if G.has_edge(gA, gB):
#             G[gA][gB]['weight'] += weight
#         else:
#             G.add_edge(gA, gB, weight=weight)


In [ ]:
import networkx as nx
import numpy as np

G = nx.Graph()

for _, row in gene_pairs_df.iterrows():
    g1 = row["Gene1"]
    g2 = row["Gene2"]

    # --- build weight ---
    pval = row["pvalue"]

    # convert p-value → weight
    weight = -np.log10(pval + 1e-12)

    # --- FIX: enforce non-negative ---
    if not np.isfinite(weight):
        continue

    if weight < 0:
        weight = 0.0

    if g1 == g2:
        continue

    G.add_edge(g1, g2, weight=weight)

NameError: name 'gene_pairs_df' is not defined

## 6. Save Graph

In [ ]:
edges = [(u, v, d['weight']) for u, v, d in G.edges(data=True)]
df_edges = pd.DataFrame(edges, columns=['gene1', 'gene2', 'weight'])
df_edges.to_csv("data/processed/gene_graph_edges.csv", index=False)

print("Saved graph with", len(df_edges), "edges")


Saved graph with 10356848 edges


## 7. Community Detection

We detect gene communities using modularity optimization:

$$
(6) \quad Q = \frac{1}{2m} \sum_{ij} \left[A_{ij} - \frac{k_i k_j}{2m}\right] \delta(c_i, c_j)
$$

We use greedy modularity maximization.

In [ ]:
# from networkx.algorithms import community

# communities = community.greedy_modularity_communities(G, weight='weight')

# # Map gene → community id
# gene_to_comm = {}
# for i, comm in enumerate(communities):
#     for gene in comm:
#         gene_to_comm[gene] = i

# print('Detected communities:', len(communities))

## 8. Visualization

We visualize the gene graph using a spring layout:

$$
(7) \quad \min \sum_{(i,j) \in E} w_{ij} ||x_i - x_j||^2
$$


In [ ]:
# import matplotlib.pyplot as plt

# pos = nx.spring_layout(G, k=0.15, iterations=20)

# plt.figure(figsize=(10, 10))

# # Draw nodes colored by community
# colors = [gene_to_comm.get(node, 0) for node in G.nodes()]

# nx.draw_networkx_nodes(G, pos, node_size=20, node_color=colors)
# nx.draw_networkx_edges(G, pos, alpha=0.1)

# plt.title('Gene Graph with Community Structure')
# plt.axis('off')
# plt.show()

## 9. Advanced Community Detection

### Louvain Method

Maximizes modularity:

$$
(8) \quad \max Q
$$

### Leiden Method

Improves Louvain by ensuring well-connected communities:

$$
(9) \quad Q_{\text{Leiden}} > Q_{\text{Louvain}}
$$

In [ ]:
# # Install if needed:
# # pip install python-louvain

# import community as community_louvain

# partition_louvain = community_louvain.best_partition(G, weight='weight')

# print("Louvain communities:", len(set(partition_louvain.values())))

ValueError: Bad node degree (-2.1653924658266332e-07)

In [ ]:
# Install if needed:
# pip install leidenalg igraph

import igraph as ig
import leidenalg

# Convert NetworkX → igraph
G_ig = ig.Graph.TupleList(
    [(u, v, d.get("weight", 1.0)) for u, v, d in G.edges(data=True)],
    weights=True,
    directed=False
)

partition_leiden = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights=G_ig.es["weight"]
)

print("Leiden communities:", len(partition_leiden))

BaseException: Could not construct partition: Cannot accept negative weights.

In [ ]:
# import matplotlib.pyplot as plt

# pos = nx.spring_layout(G, k=0.15)

# colors = [partition_louvain.get(node, 0) for node in G.nodes()]

# plt.figure(figsize=(10, 10))
# nx.draw_networkx_nodes(G, pos, node_size=20, node_color=colors)
# nx.draw_networkx_edges(G, pos, alpha=0.1)

# plt.title("Louvain Communities")
# plt.axis("off")
# plt.show()

In [ ]:
# Map Leiden membership to nodes
leiden_membership = {}
for i, comm in enumerate(partition_leiden):
    for node in comm:
        leiden_membership[G_ig.vs[node]["name"]] = i

colors = [leiden_membership.get(node, 0) for node in G.nodes()]

plt.figure(figsize=(10, 10))
nx.draw_networkx_nodes(G, pos, node_size=20, node_color=colors)
nx.draw_networkx_edges(G, pos, alpha=0.1)

plt.title("Leiden Communities")
plt.axis("off")
plt.show()